# Financial Report Standardization & Cleaning Pipeline

This notebook automates the loading, cleaning, and standardization of quarterly financial report data for Israeli companies, from Q1 2020 through full-year 2021. It reads raw JSON reports from Google Drive, unifies similar company names using fuzzy matching and manual rules, and maps them to official tickers using an Excel reference. The data is deduplicated by selecting the most complete entry per company and saved back to Drive for further analysis. While the paths reference a specific Q1 2021 example, the same pipeline is applied across all time periods consistently.


### Imports:

In [38]:
!pip install rapidfuzz

In [39]:
import pandas as pd
import numpy as np
import json

from rapidfuzz import process, fuzz
from collections import defaultdict


import re

import ast

### Mounting to drive:

In [40]:
# connect to drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Reading & presenting example file:

In [41]:
# read json file from path /content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/financial_reports_q1_2021.json with utf-8
with open('/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/financial_reports_q1_2021.json', 'r', encoding='utf-8') as f:
    financial_reports_q1_2021 = json.load(f)

In [42]:
financial_reports_q1_2021

[{'Company Name': 'Tel Aviv Stock Exchange',
  'Report Date': 'March 31, 2021',
  'Revenue': 'N/A',
  'Net Income': 'N/A',
  'Earnings Per Share (EPS)': 'N/A',
  'Gross Profit': 'N/A',
  'Operating Income (EBIT)': 'N/A',
  'Operating Cash Flow': 'N/A',
  'Investing Cash Flow': 'N/A',
  'Financing Cash Flow': 'N/A',
  'Total Assets': '890,388,000',
  'Total Liabilities': '265,772,000',
  'Short-Term Debt': 'N/A',
  'Long-Term Debt': 'N/A',
  'Shareholders’ Equity': '624,616,000',
  'Dividends Paid': '18,450,000',
  'Number of Outstanding Shares': 'N/A',
  'Guidance/Forecast': 'N/A',
  'Key Performance Indicators (KPIs)': 'N/A'},
 {'Company Name': 'S.N. Shnepp & Sons Ltd.',
  'Report Date': 'March 31, 2021',
  'Revenue': '415,370,000',
  'Net Income': '29,917,000',
  'Earnings Per Share (EPS)': '1.78',
  'Gross Profit': '108,080,000',
  'Operating Income (EBIT)': '41,995,000',
  'Operating Cash Flow': 'N/A',
  'Investing Cash Flow': 'N/A',
  'Financing Cash Flow': 'N/A',
  'Total Assets'

### Data Loading & Initial Cleaning

This function dynamically loads quarterly financial report JSON files from Google Drive, flattens nested structures, and cleans the numeric columns by removing formatting and converting values to floats. It ensures consistency across reports by standardizing the parameters per file into columns and preserving key textual fields for further processing.


In [43]:


def load_and_clean_financial_report(year: int, quarter: str):
    # Step 1: Build the file path dynamically
    path = f"/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/financial_reports_{quarter}_{year}.json"

    # Step 2: Load JSON
    with open(path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    # Step 3: Flatten list of lists
    flattened_data = []
    for item in raw_data:
        if isinstance(item, list):
            flattened_data.extend(item)
        else:
            flattened_data.append(item)

    # Step 4: Convert to DataFrame
    df = pd.DataFrame(flattened_data)

    # Step 5: Keep only first 19 columns
    df = df.iloc[:, :19]

    # Step 6: Replace 'N/A' with NaN
    df = df.replace('N/A', np.nan)

    # Step 7: Clean all columns except for preserved string ones
    string_columns = ['Company Name', 'Report Date', 'Guidance/Forecast']

    for col in df.columns:
        if col not in string_columns:
            # Clean number formatting and convert to float
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(',', '', regex=False)
                .str.replace(r'\((.*?)\)', r'-\1', regex=True)
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Step 8: Dynamically name the DataFrame
    var_name = f"financial_reports_{quarter}_{year}"
    globals()[var_name] = df
    return df


In [44]:
df2021_q1 = load_and_clean_financial_report(2021, "q1")

In [45]:
df2021_q1

,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,Tel Aviv Stock Exchange,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.903880e+08,265772000.0,NaN,NaN,624616000.0,18450000.0,NaN,NaN,NaN
1,S.N. Shnepp & Sons Ltd.,"March 31, 2021",4.153700e+08,29917000.0,1.78,108080000.0,41995000.0,NaN,NaN,NaN,5.138160e+08,200165000.0,NaN,NaN,307939000.0,10000000.0,NaN,NaN,NaN
2,Golan Plastic Products Ltd,"March 31, 2021",8.331000e+07,28681000.0,0.86,24972000.0,11974000.0,47095000.0,-30151000.0,-10281000.0,4.461790e+08,231531000.0,33800000.0,28966000.0,224978000.0,NaN,NaN,NaN,NaN
3,Orian S.M. Ltd.,"March 31, 2021",6.494000e+07,4135000.0,0.21,5510000.0,2432000.0,15264000.0,-32820000.0,30678000.0,3.019750e+08,225363000.0,8103000.0,33409000.0,76612000.0,NaN,13556686.0,NaN,NaN
4,Bikurei Sadeh (Holdings),"March 31, 2021",1.175232e+09,99133000.0,0.84,288159000.0,109095000.0,52850000.0,-20003000.0,-39082000.0,8.326480e+08,261531000.0,163594000.0,261531000.0,85381000.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423,The First International Bank Issues Ltd,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,735000000.0,NaN,NaN,NaN,NaN
424,Bank Otsar Hahayal Ltd,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8000000.0,NaN,NaN,NaN,NaN
425,Bank Otsar Hahayal Ltd,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000000.0,NaN,NaN,NaN,NaN
426,Bank Otsar Hahayal Ltd,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66000000.0,NaN,NaN,NaN,NaN


### Fuzzy Matching for Company Name Unification

This section uses fuzzy string matching to automatically group and unify variations of company names that refer to the same entity. It selects the most complete row (fewest missing values) from each group and assigns a canonical name, reducing redundancy in the dataset.


In [46]:


def auto_group_similar_names(df, threshold=80):
    company_names = df['Company Name'].dropna().unique()
    grouped = defaultdict(list)
    used = set()

    for name in company_names:
        if name in used:
            continue
        # Find close matches
        matches = process.extract(name, company_names, scorer=fuzz.token_sort_ratio)
        group = [match for match, score, _ in matches if score >= threshold]
        for match in group:
            used.add(match)
        grouped[name].extend(group)

    return grouped

def unify_companies_by_fuzzy_match(df, threshold=80):
    name_mapping = auto_group_similar_names(df, threshold)
    unified_rows = []

    for canonical_name, variations in name_mapping.items():
        subset = df[df['Company Name'].isin(variations)]
        if subset.empty:
            continue
        unified = subset.bfill().iloc[0]
        unified['Company Name'] = canonical_name
        unified_rows.append(unified)

    return pd.DataFrame(unified_rows)

# 👇 Run this
df_unified = unify_companies_by_fuzzy_match(df2021_q1, threshold=80)


<ipython-input-46-5c3039ff3f2b>:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  unified = subset.bfill().iloc[0]


In [47]:
df_unified

,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,Tel Aviv Stock Exchange,"March 31, 2021",7.840000e+07,9661000.0,0.093,NaN,NaN,41800000.0,-14700000.0,-2300000.0,8.903880e+08,265772000.0,NaN,NaN,624616000.0,18450000.0,NaN,NaN,NaN
1,S.N. Shnepp & Sons Ltd.,"March 31, 2021",4.153700e+08,29917000.0,1.780,108080000.0,41995000.0,NaN,NaN,NaN,5.138160e+08,200165000.0,NaN,NaN,307939000.0,10000000.0,NaN,NaN,NaN
2,Golan Plastic Products Ltd,"March 31, 2021",8.331000e+07,28681000.0,0.860,24972000.0,11974000.0,47095000.0,-30151000.0,-10281000.0,4.461790e+08,231531000.0,33800000.0,28966000.0,224978000.0,NaN,NaN,NaN,NaN
3,Orian S.M. Ltd.,"March 31, 2021",6.494000e+07,4135000.0,0.210,5510000.0,2432000.0,15264000.0,-32820000.0,30678000.0,3.019750e+08,225363000.0,8103000.0,33409000.0,76612000.0,NaN,13556686.0,NaN,NaN
4,Bikurei Sadeh (Holdings),"March 31, 2021",1.175232e+09,99133000.0,0.840,288159000.0,109095000.0,52850000.0,-20003000.0,-39082000.0,8.326480e+08,261531000.0,163594000.0,261531000.0,85381000.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,BioView Ltd.,"March 31, 2021",7.435000e+06,884000.0,0.060,4028000.0,NaN,730000.0,-313000.0,-1600000.0,3.388900e+07,8820000.0,NaN,NaN,25069000.0,NaN,NaN,NaN,NaN
414,Aylon Bituach Hanpakot Vegiusi Hon LTD,"March 31, 2021",3.906000e+06,-1071000.0,NaN,NaN,NaN,-24000.0,NaN,NaN,1.237850e+08,110139000.0,1467000.0,108672000.0,13646000.0,NaN,NaN,NaN,NaN
415,The First International Bank of Israel Ltd,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,927000000.0,NaN,NaN,NaN,NaN
424,Bank Otsar Hahayal Ltd,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8000000.0,NaN,NaN,NaN,NaN


In [48]:


# Load the mapping Excel file (ticker → company name)
mapping_path = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Noam's work/Best model per stock/Helpful files /company_name_to_ticker.xlsx"
mapping_df = pd.read_excel(mapping_path)
company_map = mapping_df[['Ticker', 'CompanyName']].dropna()
reference_names = company_map['CompanyName'].str.upper().unique().tolist()

# Manual name override dictionary
manual_name_map = {
    k.upper(): v.upper()
    for k, v in {
        "A. Luzon Real Estate and Finance": "LUZON GROUP",
        "Alon Blue Square Israel": "BLUE SQUARE REAL ESTATE",
        "Accel Solutions Group": "ACCEL",
        "Electreon Wireless": "ELECTREON",
        "El Al Israel Airlines.": "EL AL",
        "Elbit Medical Technologies": "ELBIT MEDITEC",
        "Enlight Renewable Energy": "ENLIGHT ENERGY",
        "Enlivex Therapeutics": "ENLIVEX",
        "Alony Hetz Properties and Investments": "ALONY HETZ",
        "Aylon Bituach Hanpakot Vegiusi Hon LTD": "AYALON HOLD.",
        "Avgol Industries 1953": "AVGOL",
        "Bet Shemesh Engines Holdings (1997) LTD.": "BET SHEMESH",
        "Bicuri Sadeh": "BIKUREY HASHDE",
        "Bikurei Sadeh (Holdings)": "BIKUREY HASHDE",
        "Biolight Life Sciences": "BIOLIGHT",
        "C.I. Systems (Israel)": "C I SYSTEMS",
        "Cellcom Israel": "CELLCOM",
        "Clal Insurance Enterprises Holdings": "CLAL INSURANCE",
        "Delek Motors Vehicle Systems": "DELEK AUTOMOTIV"

    }.items()
}

GENERIC_SUFFIXES = ['HOLDINGS', 'INVESTMENTS', 'PROPERTIES', 'GROUP', 'REAL ESTATE', 'COMPANY', 'PROPERTIES AND INVESTMENTS', 'SOLUTIONS GROUP',
                    'LIFE SCIENCES','REAL ESTATE', 'REAL ESTATE AND FINANCE', 'INDUSTRIAL DEVELOPMENT', 'ENGINEERING','COMPUTING', 'INTERNET']

def generate_variants(name: str):
    """
    Generate multiple variants of a cleaned name:
    - base form
    - base + each generic suffix
    - base - each generic word (if it exists)
    """
    base = clean_name(name)
    variants = set([base])

    # If base already contains suffixes, try removing them
    words = base.split()
    without_suffix = ' '.join([w for w in words if w not in GENERIC_SUFFIXES])
    variants.add(without_suffix)

    # Try adding common suffixes
    for suffix in GENERIC_SUFFIXES:
        variants.add(f"{base} {suffix}")
        variants.add(f"{without_suffix} {suffix}")

    return variants


# Clean helper
def clean_name(name: str) -> str:
    name = name.upper()
    name = re.sub(r'\b(LTD.|INC|LTD|CORP|LIMITED|\.|,|\&)\b', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Fuzzy matcher using cleaned names
def match_company_name_fuzzy(name: str, reference_names, threshold=85):
    # 1. Manual override first
    name_cleaned = clean_name(name)
    if name_cleaned in manual_name_map:
        return manual_name_map[name_cleaned]

    # 2. Clean all reference names into a searchable map
    ref_cleaned_map = {clean_name(ref): ref for ref in reference_names}

    # 3. Try all name variants
    name_variants = generate_variants(name)
    best_match, best_score = None, -1

    for variant in name_variants:
        match, score, _ = process.extractOne(variant, list(ref_cleaned_map.keys()), scorer=fuzz.token_sort_ratio)
        if score > best_score and score >= threshold:
            best_score = score
            best_match = ref_cleaned_map[match]

    return best_match


# Apply to your unified DataFrame
def standardize_company_names(df, reference_names, threshold=85):
    updated = []
    df["Company Name"] = df["Company Name"].str.replace(r"\s*Ltd\.?", "", regex=True).str.strip()
    df["Company Name"] = df["Company Name"].str.replace(r"Bank", "", regex=True).str.strip()


    for original in df['Company Name']:
        match = match_company_name_fuzzy(original, reference_names, threshold)
        updated_name = match if match else original  # use original if no match
        updated.append(updated_name)

    df = df.copy()
    df['Company Name'] = updated
    return df

# 👇 Run this on df_unified
df_final = standardize_company_names(df_unified, reference_names, threshold = 85)

# # Optional: Save to file
# output_path = "/content/drive/MyDrive/df_financial_reports_q1_2023_full_names.csv"
# df_final.to_csv(output_path, index=False)
# print("✅ Cleaned DataFrame saved to:", output_path)


In [49]:
df_final

,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,Tel Aviv Stock Exchange,"March 31, 2021",7.840000e+07,9661000.0,0.093,NaN,NaN,41800000.0,-14700000.0,-2300000.0,8.903880e+08,265772000.0,NaN,NaN,624616000.0,18450000.0,NaN,NaN,NaN
1,S.N. Shnepp & Sons,"March 31, 2021",4.153700e+08,29917000.0,1.780,108080000.0,41995000.0,NaN,NaN,NaN,5.138160e+08,200165000.0,NaN,NaN,307939000.0,10000000.0,NaN,NaN,NaN
2,Golan Plastic Products,"March 31, 2021",8.331000e+07,28681000.0,0.860,24972000.0,11974000.0,47095000.0,-30151000.0,-10281000.0,4.461790e+08,231531000.0,33800000.0,28966000.0,224978000.0,NaN,NaN,NaN,NaN
3,Orian S.M.,"March 31, 2021",6.494000e+07,4135000.0,0.210,5510000.0,2432000.0,15264000.0,-32820000.0,30678000.0,3.019750e+08,225363000.0,8103000.0,33409000.0,76612000.0,NaN,13556686.0,NaN,NaN
4,BIKUREY HASHDE,"March 31, 2021",1.175232e+09,99133000.0,0.840,288159000.0,109095000.0,52850000.0,-20003000.0,-39082000.0,8.326480e+08,261531000.0,163594000.0,261531000.0,85381000.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,BIO VIEW,"March 31, 2021",7.435000e+06,884000.0,0.060,4028000.0,NaN,730000.0,-313000.0,-1600000.0,3.388900e+07,8820000.0,NaN,NaN,25069000.0,NaN,NaN,NaN,NaN
414,Aylon Bituach Hanpakot Vegiusi Hon LTD,"March 31, 2021",3.906000e+06,-1071000.0,NaN,NaN,NaN,-24000.0,NaN,NaN,1.237850e+08,110139000.0,1467000.0,108672000.0,13646000.0,NaN,NaN,NaN,NaN
415,The First International of Israel,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,927000000.0,NaN,NaN,NaN,NaN
424,Otsar Hahayal,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8000000.0,NaN,NaN,NaN,NaN


### Standardizing Company Names via Mapping and Fuzzy Variants

This stage refines company names by matching them to official names from a ticker mapping Excel file using both a manual override dictionary and a robust fuzzy matching system. It generates multiple name variants (e.g., with/without suffixes) and uses token-based similarity scoring to find the best match. This ensures alignment between report names and official tickers, which is crucial for downstream financial analysis.


In [50]:

# Load mapping Excel (already loaded earlier as mapping_df)
mapping_df = pd.read_excel(mapping_path)
mapping_df = mapping_df[['CompanyName', 'Ticker', 'JsonNames']].fillna("")

# Build alias dictionary: maps every known JSON name to the official CompanyName
json_name_to_real = {}

for _, row in mapping_df.iterrows():
    real_name = str(row['CompanyName']).strip()
    json_names = row['JsonNames']

    aliases = set()

    # Try parsing as a list of names
    if isinstance(json_names, str):
        try:
            parsed = ast.literal_eval(json_names)
            if isinstance(parsed, list):
                aliases.update([x.strip().upper() for x in parsed])
            else:
                aliases.add(json_names.strip().upper())
        except:
            aliases.add(json_names.strip().upper())

    # Add mapping from each alias to the official company name
    for alias in aliases:
        if alias:
            json_name_to_real[alias] = real_name.upper()

    # Also map the official name to itself
    json_name_to_real[real_name.upper()] = real_name.upper()

# Now apply the mapping to the unified DataFrame
unified_df = df_unified.copy()
unified_df['Company Name'] = unified_df['Company Name'].astype(str).str.strip()

standardized_rows = []
not_found = []

grouped = unified_df.groupby('Company Name')

for name, group in grouped:
    key = name.strip().upper()

    # Step 1: Try to find matching canonical name
    if key in json_name_to_real:
        real_name = json_name_to_real[key]
    else:
        not_found.append(name)
        standardized_rows.append(group.iloc[0])  # keep as-is
        continue

    # Step 2: Get all rows corresponding to this real name
    aliases = [k for k, v in json_name_to_real.items() if v == real_name]
    matching_rows = unified_df[unified_df['Company Name'].str.upper().isin(aliases)]

    # Step 3: Merge info – use row with fewest NaNs
    merged_row = matching_rows.loc[matching_rows.isna().sum(axis=1).idxmin()].copy()
    merged_row['Company Name'] = real_name
    standardized_rows.append(merged_row)

# Final result
df_final_cleaned = pd.DataFrame(standardized_rows).reset_index(drop=True)

df_final_cleaned


,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,LUZON GROUP,"March 31, 2021",NaN,-671000.00,-0.420,NaN,NaN,-2.738000e+06,922000.00,2.123000e+06,1.196170e+08,3.746600e+07,1.372500e+07,3.500000e+05,8.180100e+07,NaN,NaN,NaN,NaN
1,ADAMA Agricultural Solutions,"March 31, 2021",1.020000e+09,34000000.00,NaN,285000000.0,76000000.00,-1.170000e+08,-73000000.00,2.361120e+08,6.200215e+09,3.727711e+09,5.381044e+08,1.419233e+09,2.473000e+09,NaN,137990881.0,NaN,NaN
2,ARKO Corp.,"March 31, 2021",1.484356e+09,-14662000.00,-0.130,NaN,13239000.00,1.127600e+07,-16645000.00,-8.314300e+07,2.650554e+09,2.371232e+09,2.949500e+07,6.447640e+08,1.794690e+08,NaN,124428000.0,NaN,NaN
3,ACCEL,"March 31, 2021",2.117600e+07,297000.00,0.001,4179000.0,NaN,-4.652000e+06,-8180000.00,2.013500e+07,1.296330e+08,7.807100e+07,6.080000e+05,1.316000e+06,6.890500e+07,NaN,NaN,NaN,NaN
4,Ace Capital Retail (2016),"March 31, 2021",1.794160e+08,8532000.00,0.370,80074000.0,14356000.00,NaN,NaN,NaN,8.122480e+08,6.703890e+08,NaN,NaN,1.418590e+08,20300000.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350,"Zhuhai ACCESS Semiconductor Co.,","March 31, 2021",2.974263e+08,36510319.75,NaN,NaN,38801318.25,4.614806e+07,-21847834.12,2.460594e+07,2.013603e+09,9.868398e+08,7.397944e+07,4.394626e+08,1.026763e+09,NaN,694810165.0,NaN,NaN
351,Zim Urban by Rani Zim,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
352,Ziviel Investments,"March 31, 2021",1.906000e+07,-1595000.00,NaN,4438000.0,4438000.00,1.119700e+07,-106000.00,-6.061000e+06,3.761870e+08,2.564340e+08,1.258000e+07,1.838720e+08,1.197530e+08,NaN,NaN,NaN,NaN
353,ZUR,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,5.410500e+08,76227000.00,8.118320e+08,1.529148e+10,8.154560e+09,NaN,NaN,1.802194e+09,8000000.0,64954008.0,NaN,NaN


In [51]:
# print("🔍 Companies not found in Excel mapping:")
# for name in not_found_unique:
#     print("-", name)


### Mapping JSON Aliases to Official Company Names

This block parses the `JsonNames` column from the Excel mapping file to extract known name aliases for each company. It builds a clear mapping from each alias to its official company name, helping resolve inconsistencies between report data and the reference ticker list. The result is printed in a readable format for verification.


In [52]:
# Create and print mapping: alias from 'JsonNames' → official company name
print("\n📌 JSON Alias Name → Official Company Name Mapping (from JsonNames column):\n")

alias_to_real = []

for _, row in mapping_df.iterrows():
    real_name = str(row['CompanyName']).strip()
    json_names = row['JsonNames']

    if isinstance(json_names, str) and json_names:
        try:
            parsed = ast.literal_eval(json_names)
            if isinstance(parsed, list):
                for alias in parsed:
                    alias = alias.strip()
                    if alias:
                        alias_to_real.append(f"'{alias}' --> '{real_name}'")
            else:
                alias = json_names.strip()
                if alias:
                    alias_to_real.append(f"'{alias}' --> '{real_name}'")
        except:
            alias = json_names.strip()
            if alias:
                alias_to_real.append(f"'{alias}' --> '{real_name}'")

# Sort alphabetically for readability
alias_to_real = sorted(set(alias_to_real))

for line in alias_to_real:
    print(line)



📌 JSON Alias Name → Official Company Name Mapping (from JsonNames column):

'"בזק" החברה הישראלית לתקשורת בע"מ' --> 'BEZEQ'
'(Y.Z) Queenco' --> 'QUEENCO'
'A. Luzon Real Estate and Finance' --> 'LUZON GROUP'
'A.I. Systems Conversation' --> 'AI SYSTEMS'
'A.S. Australia Israel Holdings' --> 'AUSTRALIA ISR'
'ALBAAD Massuot Yitzhak' --> 'ALBAAD'
'Abra Technologies Information' --> 'ABRA'
'Abra Technologies' --> 'ABRA'
'Accel Solutions Group' --> 'ACCEL'
'Adgar Investments & Development LTD.' --> 'ADGAR INVESTMENTS'
'Adgar Investments and Development' --> 'ADGAR INVESTMENTS'
'Afi Capital Nadlan' --> 'AFI PROPERTIES'
'Africa Israel Residences Ltd' --> 'AFRICA ISRAEL RESIDENCES'
'Airport City Ltd.' --> 'AIRPORT CITY'
'Airtouch Solar' --> 'AIRTOUCH'
'Alarum Technologies' --> 'ALARUM'
'Albad Masuot Yitzhak' --> 'ALBAAD'
'Allot Ltd.' --> 'ALLOT'
'Almeda Ventures Limited Partnership' --> 'ALMEDA PARTICIPATION UNIT'
'Almeda Ventures' --> 'ALMEDA PARTICIPATION UNIT'
'Almogim holdings' --> 'ALMOGIM'

In [53]:
df_final_cleaned

,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,LUZON GROUP,"March 31, 2021",NaN,-671000.00,-0.420,NaN,NaN,-2.738000e+06,922000.00,2.123000e+06,1.196170e+08,3.746600e+07,1.372500e+07,3.500000e+05,8.180100e+07,NaN,NaN,NaN,NaN
1,ADAMA Agricultural Solutions,"March 31, 2021",1.020000e+09,34000000.00,NaN,285000000.0,76000000.00,-1.170000e+08,-73000000.00,2.361120e+08,6.200215e+09,3.727711e+09,5.381044e+08,1.419233e+09,2.473000e+09,NaN,137990881.0,NaN,NaN
2,ARKO Corp.,"March 31, 2021",1.484356e+09,-14662000.00,-0.130,NaN,13239000.00,1.127600e+07,-16645000.00,-8.314300e+07,2.650554e+09,2.371232e+09,2.949500e+07,6.447640e+08,1.794690e+08,NaN,124428000.0,NaN,NaN
3,ACCEL,"March 31, 2021",2.117600e+07,297000.00,0.001,4179000.0,NaN,-4.652000e+06,-8180000.00,2.013500e+07,1.296330e+08,7.807100e+07,6.080000e+05,1.316000e+06,6.890500e+07,NaN,NaN,NaN,NaN
4,Ace Capital Retail (2016),"March 31, 2021",1.794160e+08,8532000.00,0.370,80074000.0,14356000.00,NaN,NaN,NaN,8.122480e+08,6.703890e+08,NaN,NaN,1.418590e+08,20300000.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350,"Zhuhai ACCESS Semiconductor Co.,","March 31, 2021",2.974263e+08,36510319.75,NaN,NaN,38801318.25,4.614806e+07,-21847834.12,2.460594e+07,2.013603e+09,9.868398e+08,7.397944e+07,4.394626e+08,1.026763e+09,NaN,694810165.0,NaN,NaN
351,Zim Urban by Rani Zim,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
352,Ziviel Investments,"March 31, 2021",1.906000e+07,-1595000.00,NaN,4438000.0,4438000.00,1.119700e+07,-106000.00,-6.061000e+06,3.761870e+08,2.564340e+08,1.258000e+07,1.838720e+08,1.197530e+08,NaN,NaN,NaN,NaN
353,ZUR,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,5.410500e+08,76227000.00,8.118320e+08,1.529148e+10,8.154560e+09,NaN,NaN,1.802194e+09,8000000.0,64954008.0,NaN,NaN


### Alias Resolution and Final Deduplication

This final cleaning phase ensures that company names appearing as aliases (from the `JsonNames` column) are mapped to their official names using a pre-built alias dictionary. After resolving all aliases, the dataset is filtered to retain only companies listed in the Excel mapping. For each matched company, the most complete record (least missing values) is selected, resulting in a clean and deduplicated DataFrame ready for export.


In [54]:
# Step 0: Normalize alias_to_real into a usable map
# (Assuming alias_to_real was built earlier as a list of "'alias' --> 'real_name'" strings)
alias_dict = {}
for line in alias_to_real:
    try:
        alias, real = line.split("-->")
        alias = alias.strip().strip("'").upper()
        real = real.strip().strip("'").upper()
        alias_dict[alias] = real
    except:
        continue

# Step 1: Replace aliases in df_final_cleaned using alias_dict
df_with_resolved_aliases = df_final_cleaned.copy()
df_with_resolved_aliases['Company Name'] = df_with_resolved_aliases['Company Name'].str.upper().str.strip()
df_with_resolved_aliases['Company Name'] = df_with_resolved_aliases['Company Name'].apply(
    lambda name: alias_dict.get(name, name)
)

# Step 2: Get all real company names from Excel
real_company_names_from_excel = mapping_df['CompanyName'].dropna().str.upper().unique().tolist()

# Step 3: Filter to include only companies that match Excel names after alias resolution
filtered_df = df_with_resolved_aliases[
    df_with_resolved_aliases['Company Name'].isin(real_company_names_from_excel)
].copy()

# Step 4: For each company, keep the row with the fewest NaNs
best_rows = []
for company_name in filtered_df['Company Name'].unique():
    group = filtered_df[filtered_df['Company Name'] == company_name]
    best_row = group.loc[group.isna().sum(axis=1).idxmin()]
    best_rows.append(best_row)

# Step 5: Final deduplicated DataFrame
df_final_deduped = pd.DataFrame(best_rows).reset_index(drop=True)

# Show result
print("✅ Deduplicated DataFrame shape:", df_final_deduped.shape)
df_final_deduped


✅ Deduplicated DataFrame shape: (188, 19)


,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,LUZON GROUP,"March 31, 2021",NaN,-671000.0,-0.420,NaN,NaN,-2738000.0,922000.0,2123000.0,1.196170e+08,3.746600e+07,1.372500e+07,3.500000e+05,8.180100e+07,NaN,NaN,NaN,NaN
1,ACCEL,"March 31, 2021",21176000.0,297000.0,0.001,4179000.0,NaN,-4652000.0,-8180000.0,20135000.0,1.296330e+08,7.807100e+07,6.080000e+05,1.316000e+06,6.890500e+07,NaN,NaN,NaN,NaN
2,AFI PROPERTIES,2021-03-31,NaN,50300000.0,NaN,NaN,NaN,NaN,NaN,NaN,1.364527e+10,NaN,NaN,NaN,4.280821e+09,NaN,NaN,NaN,NaN
3,AIRPORT CITY,31.3.2021,NaN,191952000.0,1.500,610012000.0,337388000.0,NaN,NaN,NaN,1.526052e+10,7.580465e+09,1.329952e+09,4.041292e+09,7.477794e+09,NaN,NaN,The Company cannot estimate the full impact of...,NaN
4,ALBAAD,"March 31, 2021",371345000.0,32419000.0,3.050,97596000.0,43640000.0,NaN,NaN,NaN,1.482447e+09,9.654570e+08,NaN,NaN,5.169900e+08,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,YBOX,"March 31, 2021",130650000.0,25364000.0,0.130,21030000.0,39034000.0,6470000.0,-4141000.0,-1501000.0,6.367670e+08,4.171630e+08,1.461170e+08,1.929830e+08,2.196040e+08,NaN,NaN,NaN,NaN
184,YAACOBI GROUP,"March 31, 2021",106508000.0,252000.0,NaN,11287000.0,1468000.0,NaN,5712000.0,-32056000.0,5.059090e+08,2.463950e+08,5.894700e+07,2.914300e+07,1.959800e+08,NaN,NaN,The company estimates that in 2022 and after c...,NaN
185,ISRAS,"March 31, 2021",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.208940e+06,3.967830e+06,6.877010e+05,NaN,3.553409e+06,NaN,NaN,NaN,NaN
186,UNET CREDIT,"March 31, 2021",12742000.0,-11395000.0,-2.560,NaN,NaN,-31597000.0,-2372000.0,69991000.0,1.158610e+08,8.507400e+07,4.796000e+07,4.239600e+07,3.078700e+07,NaN,NaN,NaN,NaN


### Exporting to drive

In [55]:
df_final_deduped.to_csv("/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/report_analysis_2021_q1.csv")